# Central Virginia Tree Canopy — Policy Modeling Data Pipeline

**Purpose:** Download and merge four external policy datasets with GEDI canopy data
to produce a panel dataset suitable for multivariate / fixed-effects regression:

$$Y_{it} = \beta_0 + \beta_1 C_{it} + \beta_2 X_{it} + \alpha_i + \gamma_t + \varepsilon_{it}$$

| Symbol | Meaning |
|---|---|
| $Y$ | Outcome: K-12 SOL pass rate, violent crime total, or health outcome % |
| $C$ | Canopy: `mean_canopy_cover` (GEDI L2B) or `canopy_height_mean_m` (GEDI L2A) |
| $X$ | Controls: median household income, population density, % bachelor's degree+ |
| $\alpha_i$ | Jurisdiction fixed effect |
| $\gamma_t$ | Year fixed effect |

**Data Sources:**
1. **GEDI L2A + L2B** — `s3://central-virginia-tree-canopy-project/dashboard-data/`
2. **VDOE SOL** — Virginia Open Data Portal + VDOE direct XLS (2019–2025)
3. **Virginia Crime** — FBI UCR Table 10 (2019) + VSP Annual Reports (2020–2024)
4. **CDC PLACES** — data.cdc.gov Socrata API (2024 release)
5. **Census ACS 5-Year** — api.census.gov (income, education, population density)

**Study Jurisdictions:** Albemarle, Augusta, Charlottesville, Fluvanna, Greene, Louisa, Nelson, Rockingham

**Study Years:** 2019–2025

## Cell 1 — Install Dependencies
Run once per kernel session. Restart the kernel after installation.

In [1]:
import subprocess, sys

# Pin the boto stack to a mutually compatible set (required for SageMaker)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall",
    "aiobotocore==3.7.0",
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
])

# Install all other dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "aioitertools",
    "openpyxl",
    "xlrd",
    "tabula-py",
    "linearmodels",
    "statsmodels",
    "jpype1",
    "sodapy",
    "tqdm",
])

print("All dependencies installed. Restart the kernel if this is the first run.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.42 requires botocore==1.43.42, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.42 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


All dependencies installed. Restart the kernel if this is the first run.


## Cell 2 — Imports and Configuration

In [28]:
import json
import logging
import os
import time
import warnings
from pathlib import Path

import boto3
import pandas as pd
import requests

import s3fs
import re

from sodapy import Socrata

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── S3 Configuration ──────────────────────────────────────────────────────────
S3_BUCKET          = "central-virginia-tree-canopy-project"
S3_DASHBOARD_PREFIX = "dashboard-data/"
GEDI_L2A_KEY       = "dashboard-data/gedi_county_summary.json"
GEDI_L2B_KEY       = "dashboard-data/gedi02B_county_summary.json"
CRIME_S3_KEY       = "data/multivariate/virginia_crime_county_year.csv"

# ── Local output directory (SageMaker) ────────────────────────────────────────
OUTPUT_DIR = Path("/home/ec2-user/SageMaker/policy_pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Census API key (set as SageMaker environment variable or paste here) ──────
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "1d33172c35f206132f94362ee4786af3e6a82a62")
if not CENSUS_API_KEY:
    log.warning(
        "CENSUS_API_KEY not set. ACS requests may be rate-limited.\n"
        "Get a free key at: https://api.census.gov/data/key_signup.html\n"
        "Set it with: import os; os.environ['CENSUS_API_KEY'] = 'your_key'"
    )

# ── Study-area constants ──────────────────────────────────────────────────────
DIVISION_CODES = {
    "Albemarle":       "002",
    "Augusta":         "015",
    "Charlottesville": "104",
    "Fluvanna":        "044",
    "Greene":          "055",
    "Louisa":          "075",
    "Nelson":          "090",
    "Orange":          "068",
    "Rockingham":      "137",
}

FIPS_CODES = {
    "Albemarle":       "51003",
    "Augusta":         "51015",
    "Charlottesville": "51540",
    "Fluvanna":        "51065",
    "Greene":          "51079",
    "Louisa":          "51109",
    "Nelson":          "51125",
    "Orange":          "51137",
    "Rockingham":      "51165",
}

HEALTH_MEASURES = [
    "BPHIGH", "CASTHMA", "CSMOKING", "DEPRESSION",
    "DIABETES", "MHLTH", "OBESITY", "PHLTH",
]

COUNTY_FIPS = {k: v[2:] for k, v in FIPS_CODES.items()}
STUDY_YEARS = list(range(2019, 2026))

print("Configuration loaded.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"GEDI L2A: s3://{S3_BUCKET}/{GEDI_L2A_KEY}")
print(f"GEDI L2B: s3://{S3_BUCKET}/{GEDI_L2B_KEY}")

Configuration loaded.
Output directory: /home/ec2-user/SageMaker/policy_pipeline_output
GEDI L2A: s3://central-virginia-tree-canopy-project/dashboard-data/gedi_county_summary.json
GEDI L2B: s3://central-virginia-tree-canopy-project/dashboard-data/gedi02B_county_summary.json


## Cell 3 — Helper: Download Utility

In [29]:
def download(url: str, dest: Path, label: str, retries: int = 3) -> bool:
    """Download a URL to a local file. Returns True on success."""
    for attempt in range(1, retries + 1):
        try:
            log.info(f"[{label}] Downloading (attempt {attempt}): {url}")
            resp = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            dest.write_bytes(resp.content)
            log.info(f"[{label}] Saved -> {dest}  ({len(resp.content):,} bytes)")
            return True
        except Exception as exc:
            log.warning(f"[{label}] Attempt {attempt} failed: {exc}")
            time.sleep(2 ** attempt)
    log.error(f"[{label}] All {retries} attempts failed.")
    return False

print("Helper functions defined.")

Helper functions defined.


## Cell 4 — Load GEDI Data from S3

Loads `gedi_county_summary.json` (L2A — canopy height) and
`gedi02B_county_summary.json` (L2B — canopy cover) from
`s3://central-virginia-tree-canopy-project/dashboard-data/`
and merges them into a single GEDI panel.

In [30]:
s3 = boto3.client("s3")

def load_json_from_s3(bucket: str, key: str, label: str) -> pd.DataFrame:
    """Download a JSON file from S3 and return it as a DataFrame."""
    local_path = OUTPUT_DIR / Path(key).name
    log.info(f"[{label}] Downloading s3://{bucket}/{key}")
    s3.download_file(bucket, key, str(local_path))
    data = json.loads(local_path.read_text())
    df = pd.DataFrame(data)
    log.info(f"[{label}] Loaded {len(df)} rows, columns: {list(df.columns)}")
    return df

# Load GEDI L2A (canopy height)
df_l2a = load_json_from_s3(S3_BUCKET, GEDI_L2A_KEY, "GEDI-L2A")

# Load GEDI L2B (canopy cover)
df_l2b = load_json_from_s3(S3_BUCKET, GEDI_L2B_KEY, "GEDI-L2B")

# Merge L2A and L2B on jurisdiction + year
l2a_cols = [c for c in ["jurisdiction", "year", "canopy_height_mean_m"] if c in df_l2a.columns]
l2b_cols = [c for c in ["jurisdiction", "year", "mean_canopy_cover", "total_valid_shots"] if c in df_l2b.columns]

gedi = pd.merge(
    df_l2a[l2a_cols],
    df_l2b[l2b_cols],
    on=["jurisdiction", "year"],
    how="outer",
)
gedi["jurisdiction"] = gedi["jurisdiction"].str.strip()
gedi["year"] = pd.to_numeric(gedi["year"], errors="coerce").astype("Int64")

print(f"\nGEDI panel: {len(gedi)} rows x {len(gedi.columns)} columns")
print(f"Jurisdictions: {sorted(gedi['jurisdiction'].unique())}")
print(f"Years: {sorted(gedi['year'].dropna().unique())}")
gedi.head(10)

00:21:01  INFO      [GEDI-L2A] Downloading s3://central-virginia-tree-canopy-project/dashboard-data/gedi_county_summary.json
00:21:01  INFO      [GEDI-L2A] Loaded 68 rows, columns: ['year', 'jurisdiction', 'canopy_height_mean_m']
00:21:01  INFO      [GEDI-L2B] Downloading s3://central-virginia-tree-canopy-project/dashboard-data/gedi02B_county_summary.json
00:21:01  INFO      [GEDI-L2B] Loaded 68 rows, columns: ['jurisdiction', 'year', 'mean_canopy_cover', 'total_valid_shots']



GEDI panel: 68 rows x 5 columns
Jurisdictions: ['Albemarle', 'Augusta', 'Buckingham', 'Charlottesville', 'Fluvanna', 'Greene', 'Louisa', 'Nelson', 'Orange', 'Rockingham']
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,jurisdiction,year,canopy_height_mean_m,mean_canopy_cover,total_valid_shots
0,Albemarle,2019,19.675812,0.529159,53356
1,Albemarle,2020,18.465603,0.540116,14835
2,Albemarle,2021,18.793875,0.424221,54570
3,Albemarle,2022,18.639362,0.459664,47618
4,Albemarle,2023,13.609988,0.207803,11110
5,Albemarle,2024,20.382057,0.512670,31573
6,Albemarle,2025,17.832790,0.404998,27296
7,Augusta,2019,14.742975,0.412462,23961
8,Augusta,2020,10.717906,0.233118,28070
9,Augusta,2021,14.507577,0.374634,33316


## Cell 5 — Download VDOE SOL Pass Rates

Downloads Division Subject-Area XLS files from the VDOE website for school
years 2018-19 through 2024-25 and parses them into a tidy DataFrame.

In [31]:
# ── S3 location for manually uploaded VDOE file ───────────────────────────────
VDOE_S3_KEY = "data/multivariate/vaDeptOfEduAssessmentStatistics.csv"

NAME_TO_JURIS = {
    "albemarle county":       "Albemarle",
    "albemarle co.":          "Albemarle",
    "augusta county":         "Augusta",
    "augusta co.":            "Augusta",
    "charlottesville city":   "Charlottesville",
    "charlottesville":        "Charlottesville",
    "fluvanna county":        "Fluvanna",
    "fluvanna co.":           "Fluvanna",
    "greene county":          "Greene",
    "greene co.":             "Greene",
    "louisa county":          "Louisa",
    "louisa co.":             "Louisa",
    "nelson county":          "Nelson",
    "nelson co.":             "Nelson",
    "rockingham county":      "Rockingham",
    "rockingham co.":         "Rockingham",
}

def load_vdoe_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """Load the VDOE SOL CSV from S3 and return a tidy jurisdiction × year DataFrame."""

    fs = s3fs.S3FileSystem(anon=False)
    
    # Load SMAP annual summary (region-wide, one row per year)
    vdoe_s3_path = f"s3://{bucket}/{key}"
    print(f"Loading [VDOE] from : {vdoe_s3_path}")
    with fs.open(vdoe_s3_path, "rb") as f:
        vdoe_df = pd.read_csv(f)
    
    #df = pd.read_csv(local_path, encoding="utf-8", low_memory=False)
    log.info(f"[VDOE] Raw shape: {vdoe_df.shape}, columns: {list(vdoe_df.columns[:10])}")

    # Normalize column names
    vdoe_df.columns = (
        vdoe_df.columns.astype(str).str.strip().str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # Identify key columns flexibly
    #div_col  = next((c for c in vdoe_df.columns if "division" in c or "district" in c), None)
    div_col = next(
        (c for c in vdoe_df.columns if re.search(r"division.?name", c, re.I)), None
    ) or next(
        (c for c in vdoe_df.columns if "division" in c.lower() and "number" not in c.lower()), None
    )
    yr_col   = next((c for c in vdoe_df.columns if "year" in c or "school_year" in c), None)
    rate_col = next((c for c in vdoe_df.columns
                     if ("pass" in c and "rate" in c)
                     or ("pct" in c and ("pass" in c or "proficient" in c))
                     or c in ("pass_rate", "pass_rate_all", "pct_pass")), None)
    subj_col = next((c for c in vdoe_df.columns if "subject" in c or "test_name" in c
                     or "content_area" in c), None)

    log.info(f"[VDOE] Identified columns — division: {div_col}, year: {yr_col}, "
             f"rate: {rate_col}, subject: {subj_col}")

    if div_col is None or rate_col is None:
        log.error(f"[VDOE] Cannot identify required columns. Available: {list(df.columns)}")
        return pd.DataFrame()

    #log.info(vdoe_df.head(10))
    
    # Map division names to study jurisdictions
    vdoe_df["jurisdiction"] = vdoe_df[div_col].astype(str).str.strip().str.lower().map(NAME_TO_JURIS)
    vdoe_df = vdoe_df.dropna(subset=["jurisdiction"])

    if vdoe_df.empty:
        log.warning("[VDOE] No matching jurisdictions found after name mapping.")
        return pd.DataFrame()

    # Filter to all-subjects rows if a subject column exists
    if subj_col:
        all_mask = vdoe_df[subj_col].astype(str).str.lower().str.contains(
            r"all|total|composite", na=False
        )
        if all_mask.sum() > 0:
            vdoe_df = vdoe_df[all_mask]
        else:
            
            vdoe_df[rate_col] = pd.to_numeric(vdoe_df[rate_col], errors="coerce")
            
            # Average across all subjects per jurisdiction × year
            vdoe_df = (
                vdoe_df.groupby(["jurisdiction"] + ([yr_col] if yr_col else []))[rate_col]
                .mean()
                .reset_index()
            )

    #vdoe_df[rate_col] = pd.to_numeric(vdoe_df[rate_col], errors="coerce")

    # Derive calendar year
    if yr_col:
        # Handle formats like "2022-23", "2023", "2022-2023"
        def parse_year(val):
            s = str(val).strip()
            if "-" in s:
                parts = s.split("-")
                last = parts[-1]
                return int(last) + 2000 if len(last) == 2 else int(last)
            try:
                return int(float(s))
            except Exception:
                return None

        vdoe_df["year"] = vdoe_df[yr_col].apply(parse_year)
    else:
        log.warning("[VDOE] No year column found — assigning year=None. "
                    "Check the CSV and add a year column manually if needed.")
        vdoe_df["year"] = None

    result = (
        vdoe_df[["jurisdiction", "year", rate_col]]
        .rename(columns={rate_col: "sol_pass_rate_all"})
        .dropna(subset=["sol_pass_rate_all"])
        .drop_duplicates(subset=["jurisdiction", "year"], keep="last")
        .sort_values(["jurisdiction", "year"])
        .reset_index(drop=True)
    )

    log.info(f"[VDOE] Parsed {len(result)} rows for {result['jurisdiction'].nunique()} jurisdictions")
    return result


sol = load_vdoe_from_s3(S3_BUCKET, VDOE_S3_KEY)

if not sol.empty:
    print(f"\nVDOE SOL dataset: {len(sol)} rows")
    print(sol.pivot_table(index="jurisdiction", columns="year", values="sol_pass_rate_all"))
else:
    print("WARNING: VDOE SOL data could not be loaded. Check the CSV structure.")
    print(f"Expected S3 location: s3://{S3_BUCKET}/{VDOE_S3_KEY}")


00:21:03  INFO      [VDOE] Raw shape: (266, 5), columns: ['School Year', 'Division Number', 'Division Name', 'Subject', 'Pass Rate']
00:21:03  INFO      [VDOE] Identified columns — division: division_name, year: school_year, rate: pass_rate, subject: subject
00:21:03  INFO      [VDOE] Parsed 48 rows for 8 jurisdictions


Loading [VDOE] from : s3://central-virginia-tree-canopy-project/data/multivariate/vaDeptOfEduAssessmentStatistics.csv

VDOE SOL dataset: 48 rows
year               2019     2021    2022    2023     2024     2025
jurisdiction                                                      
Albemarle        79.472  70.9100  67.400  71.156  73.0580  73.3380
Augusta          81.476  61.9320  71.560  66.586  62.9080  65.5160
Charlottesville  71.034  51.7380  53.898  54.928  62.7475  62.5100
Fluvanna         79.984  63.4420  67.198  65.240  71.4800  69.5740
Greene           73.464  53.4800  54.062  59.348  63.0475  66.9575
Louisa           82.046  72.4975  71.308  75.312  82.6120  85.0940
Nelson           79.828  50.0500  66.538  70.430  71.5925  74.6875
Rockingham       76.928  55.4075  60.260  60.560  65.0150  67.7250


In [32]:
# VDOE_XLS_URLS = {
#     "2018-19": "https://www.doe.virginia.gov/home/showpublisheddocument/32592/638047216260530000",
#     "2020-21": "https://www.doe.virginia.gov/home/showpublisheddocument/32568/638047216189130000",
#     "2021-22": "https://www.doe.virginia.gov/home/showpublisheddocument/32594/638047216265870000",
#     "2022-23": "https://www.doe.virginia.gov/home/showpublisheddocument/48939/638379725468370000",
#     "2023-24": "https://www.doe.virginia.gov/home/showpublisheddocument/56492/638632943529970000",
#     "2024-25": "https://www.doe.virginia.gov/home/showpublisheddocument/65231/638918222349300000",
# }

# NAME_TO_JURIS = {
#     "albemarle county":       "Albemarle",
#     "augusta county":         "Augusta",
#     "charlottesville city":   "Charlottesville",
#     "fluvanna county":        "Fluvanna",
#     "greene county":          "Greene",
#     "louisa county":          "Louisa",
#     "nelson county":          "Nelson",
#     "rockingham county":      "Rockingham",
# }

# def parse_vdoe_xls(path: Path, school_year_label: str) -> pd.DataFrame:
#     """Parse a VDOE Division Subject-Area XLS into a tidy DataFrame."""
#     try:
#         xl = pd.read_excel(path, sheet_name=0, header=None)
#     except Exception as exc:
#         log.warning(f"[VDOE] Could not parse {path.name}: {exc}")
#         return pd.DataFrame()

#     # Find header row
#     header_row = None
#     for i, row in xl.iterrows():
#         if any("division" in str(c).lower() for c in row.values):
#             header_row = i
#             break
#     if header_row is None:
#         log.warning(f"[VDOE] No header row found in {path.name}")
#         return pd.DataFrame()

#     xl.columns = (
#         xl.iloc[header_row]
#         .astype(str).str.strip().str.lower()
#         .str.replace(r"\s+", "_", regex=True)
#     )
#     xl = xl.iloc[header_row + 1:].reset_index(drop=True)

#     div_col  = next((c for c in xl.columns if "division" in c), None)
#     rate_col = next((c for c in xl.columns if "pass" in c and "rate" in c), None)
#     subj_col = next((c for c in xl.columns if "subject" in c or "test" in c), None)

#     if div_col is None or rate_col is None:
#         log.warning(f"[VDOE] Missing required columns in {path.name}: {list(xl.columns)}")
#         return pd.DataFrame()

#     keep = [div_col, rate_col] + ([subj_col] if subj_col else [])
#     xl = xl[keep].copy()
#     xl.columns = ["division_name", "pass_rate"] + (["subject"] if subj_col else [])
#     xl = xl.dropna(subset=["division_name", "pass_rate"])
#     xl["pass_rate"] = pd.to_numeric(xl["pass_rate"], errors="coerce")

#     if subj_col and "subject" in xl.columns:
#         all_subj = xl[xl["subject"].astype(str).str.lower().str.contains("all", na=False)]
#         if len(all_subj) == 0:
#             xl = xl.groupby("division_name")["pass_rate"].mean().reset_index()
#         else:
#             xl = all_subj[["division_name", "pass_rate"]].copy()

#     xl["division_name"] = xl["division_name"].astype(str).str.strip()
#     xl["jurisdiction"] = xl["division_name"].str.lower().map(NAME_TO_JURIS)
#     xl = xl.dropna(subset=["jurisdiction"])

#     # Derive calendar year from school year label (e.g. "2022-23" -> 2023)
#     yr_part = school_year_label.split("-")[1]
#     cal_year = int(yr_part) + 2000 if len(yr_part) == 2 else int(yr_part)
#     xl["year"] = cal_year

#     return xl[["jurisdiction", "year", "pass_rate"]].rename(
#         columns={"pass_rate": "sol_pass_rate_all"}
#     )


# sol_frames = []
# for label, url in VDOE_XLS_URLS.items():
#     dest = OUTPUT_DIR / f"vdoe_sol_{label.replace('-', '_')}.xlsx"
#     ok = download(url, dest, f"VDOE/{label}")
#     if ok:
#         df = parse_vdoe_xls(dest, label)
#         if not df.empty:
#             log.info(f"[VDOE] {label}: {len(df)} rows parsed")
#             sol_frames.append(df)

# if sol_frames:
#     sol = pd.concat(sol_frames, ignore_index=True)
#     sol = sol.sort_values("year").drop_duplicates(subset=["jurisdiction", "year"], keep="last")
#     print(f"\nVDOE SOL dataset: {len(sol)} rows")
#     print(sol.pivot_table(index="jurisdiction", columns="year", values="sol_pass_rate_all"))
# else:
#     sol = pd.DataFrame(columns=["jurisdiction", "year", "sol_pass_rate_all"])
#     print("WARNING: No VDOE SOL data retrieved. Check VDOE URL availability.")

## Cell 6 — Load Virginia Crime Data

In [33]:
# Load Virginia Crime Data (region-wide, one row per year)

fs = s3fs.S3FileSystem(anon=False)

CRIME_S3_KEY = "data/multivariate/virginia_crime_county_year.csv"
vcd_s3_path = f"s3://{S3_BUCKET}/{CRIME_S3_KEY}"
print(f"Loading [Virginia Crime Data] from : {vcd_s3_path}")
with fs.open(vcd_s3_path, "rb") as f:
    crime = pd.read_csv(f)

crime.head(10)

Loading [Virginia Crime Data] from : s3://central-virginia-tree-canopy-project/data/multivariate/virginia_crime_county_year.csv


,year,jurisdiction,agency_name,agency_code,population_estimate,incident_total,offense_total,group_a_crimes_per_100k,total_arrests,adult_arrests,...,shoplifting_reported,theft_from_motor_vehicle_reported,all_other_larceny_reported,drug_narcotic_violations_reported,drug_equipment_violations_reported,weapon_law_violations_reported,dui_group_b_adult,drunkenness_group_b_adult,disorderly_conduct_group_b_adult,trespass_real_property_group_b_adult
0,2019,Albemarle,Albemarle County Police Department,VA0020300,109722.0,3324.0,4042.0,3029.5,1838.0,1766.0,...,251.0,305.0,403.0,408.0,219.0,67.0,153.0,88.0,12.0,45.0
1,2019,Augusta,Augusta County Sheriff's Office,VA0080000,75831.0,2285.0,2560.0,3013.3,919.0,892.0,...,62.0,110.0,180.0,344.0,49.0,56.0,48.0,78.0,13.0,17.0
2,2019,Buckingham,Buckingham County Sheriff's Office,VA0150000,17075.0,252.0,300.0,1475.8,179.0,179.0,...,3.0,8.0,50.0,25.0,6.0,8.0,28.0,8.0,4.0,8.0
3,2019,Charlottesville,Charlottesville Police Department,VA1020000,49181.0,2485.0,2910.0,5052.8,1471.0,1445.0,...,155.0,247.0,376.0,140.0,41.0,36.0,50.0,204.0,7.0,44.0
4,2019,Fluvanna,Fluvanna County Sheriff's Office,VA0320000,27038.0,574.0,657.0,2122.9,312.0,312.0,...,6.0,22.0,96.0,81.0,8.0,17.0,50.0,0.0,6.0,1.0
5,2019,Greene,Greene County Sheriff's Office,VA0390000,20097.0,626.0,789.0,3114.9,563.0,535.0,...,91.0,12.0,48.0,100.0,65.0,9.0,25.0,21.0,8.0,9.0
6,2019,Louisa,Louisa Co. Sheriff Office11,VA0540000,34918.0,861.0,964.0,2465.8,245.0,242.0,...,38.0,61.0,130.0,111.0,5.0,31.0,15.0,25.0,2.0,4.0
7,2019,Nelson,Nelson County Sheriff's Office,VA0620000,14794.0,570.0,661.0,3852.9,167.0,134.0,...,14.0,13.0,65.0,98.0,5.0,15.0,8.0,5.0,0.0,95.0
8,2019,Orange,Orange County Sheriffs Office,VA0680000,29267.0,455.0,480.0,1554.7,249.0,246.0,...,19.0,9.0,47.0,86.0,15.0,8.0,46.0,14.0,1.0,2.0
9,2019,Rockingham,Rockingham CO Sheriff's Dept,VA0820000,62138.0,1149.0,1458.0,1849.1,1260.0,1176.0,...,30.0,48.0,100.0,279.0,154.0,40.0,71.0,89.0,5.0,31.0


## Cell 7 — Download CDC PLACES Health Data

Fetches county-level health estimates for all 8 study jurisdictions from
the CDC PLACES Socrata API (2024 release).

Key measures: obesity, diabetes, poor mental health, poor physical health, smoking.

In [34]:
# Target CDC PLACES OpenData endpoint identifier for standard county rows
DATASET_ID = "swc5-untb"

# Recommended approach: Load from system environment variables
# In your Linux terminal run: export CDC_APP_TOKEN="your_actual_token_here"
CDC_TOKEN = os.getenv("CDC_APP_TOKEN", "Ie8UYPPCPKZqsjJWbrFG3Oyv9")

# Pass your token as the second argument to bypass public throttle limits
client = Socrata("data.cdc.gov", CDC_TOKEN)

# Formulate lists into clean SQL 'IN' string arrays for the query engine
measures_in = ", ".join([f"'{m}'" for m in HEALTH_MEASURES])
fips_in     = ", ".join([f"'{f}'" for f in FIPS_CODES.values()])

# Build the SoQL filter execution string
# Filters for Virginia (VA), your specific county FIPS codes, the metrics array,
# and age-adjusted prevalence only (eliminates the duplicate crude-prevalence rows)
where_clause = (
    f"stateabbr = 'VA' AND "
    f"locationid IN ({fips_in}) AND "
    f"measureid IN ({measures_in}) AND "
    f"data_value_type = 'Age-adjusted prevalence'"
)

print("Executing batch pull from CDC PLACES API...")
try:
    # Query up to 2000 matching rows to handle multiple variables across counties
    results = client.get(DATASET_ID, where=where_clause, limit=2000)

    # Pack into a standard data analysis dataframe
    df = pd.DataFrame.from_records(results)

    if not df.empty:
        # Isolate key metrics columns
        output_cols    = ["locationname", "locationid", "measureid",
                          "data_value", "data_value_type", "year"]
        available_cols = [c for c in output_cols if c in df.columns]

        # Sort values cleanly by County Name and Metric Type
        df_clean = df[available_cols].sort_values(by=["locationname", "measureid"])
        df_clean["data_value"] = pd.to_numeric(df_clean["data_value"], errors="coerce")

        print(f"\nSuccessfully pulled {len(df_clean)} rows (age-adjusted only).")
        print(df_clean.head(15).to_string(index=False))

        # Save the raw age-adjusted rows for audit / reference
        df_clean.to_csv(OUTPUT_DIR / "central_va_cdc_places_metrics.csv", index=False)
        print(f"\nSaved raw data matrix to {OUTPUT_DIR}/central_va_cdc_places_metrics.csv")
        #print("\nSaved raw data matrix to OUTPUT_DIR '/central_va_cdc_places_metrics.csv'")

        # ── Pivot to one row per jurisdiction, one column per measure ──────────
        cdc = df_clean.pivot_table(
            index="locationname",
            columns="measureid",
            values="data_value",
            aggfunc="first",        # one value per cell after age-adjusted filter
        ).reset_index()

        cdc.columns.name = None
        cdc = cdc.rename(columns={
            "locationname": "jurisdiction",
            "BPHIGH":       "health_bphigh_pct",
            "CASTHMA":      "health_casthma_pct",
            "CSMOKING":     "health_csmoking_pct",
            "DEPRESSION":   "health_depression_pct",
            "DIABETES":     "health_diabetes_pct",
            "MHLTH":        "health_mhlth_pct",
            "OBESITY":      "health_obesity_pct",
            "PHLTH":        "health_phlth_pct",
        })
        cdc["places_data_year"] = 2023

        print(f"\nCDC PLACES panel: {len(cdc)} jurisdictions x {len(cdc.columns)} columns")
        print(cdc.to_string(index=False))

    else:
        print("API executed successfully but returned zero rows matching criteria.")
        cdc = pd.DataFrame()

except Exception as e:
    print(f"API Endpoint Execution Failed: {e}")
    cdc = pd.DataFrame()
finally:
    client.close()


Executing batch pull from CDC PLACES API...

Successfully pulled 72 rows (age-adjusted only).
locationname locationid  measureid  data_value         data_value_type year
   Albemarle      51003     BPHIGH        28.5 Age-adjusted prevalence 2023
   Albemarle      51003    CASTHMA        10.1 Age-adjusted prevalence 2023
   Albemarle      51003   CSMOKING         9.6 Age-adjusted prevalence 2023
   Albemarle      51003 DEPRESSION        21.8 Age-adjusted prevalence 2023
   Albemarle      51003   DIABETES         8.7 Age-adjusted prevalence 2023
   Albemarle      51003      MHLTH        15.2 Age-adjusted prevalence 2023
   Albemarle      51003    OBESITY        29.0 Age-adjusted prevalence 2023
   Albemarle      51003      PHLTH        10.5 Age-adjusted prevalence 2023
     Augusta      51015     BPHIGH        31.7 Age-adjusted prevalence 2023
     Augusta      51015    CASTHMA        10.5 Age-adjusted prevalence 2023
     Augusta      51015   CSMOKING        15.0 Age-adjusted prevalence

In [35]:

# # Target CDC PLACES OpenData endpoint identifier for standard county rows
# DATASET_ID = "swc5-untb"

# # Recommended approach: Load from system environment variables
# # In your Linux terminal run: export CDC_APP_TOKEN="your_actual_token_here"
# CDC_TOKEN = os.getenv("CDC_APP_TOKEN", "Ie8UYPPCPKZqsjJWbrFG3Oyv9")

# # Pass your token as the second argument to bypass public throttle limits
# client = Socrata("data.cdc.gov", CDC_TOKEN)

# #probe = client.get(DATASET_ID, where="stateabbr = 'VA'", limit=3,
# #                   select="locationname, locationid, measureid")
# #print(pd.DataFrame(probe))

# # Formulate lists into clean SQL 'IN' string arrays for the query engine
# measures_in = ", ".join([f"'{m}'" for m in HEALTH_MEASURES])
# #fips_in = ", ".join([f"'{f}'" for f in COUNTY_FIPS.values()])
# fips_in = ", ".join([f"'{f}'" for f in FIPS_CODES.values()])

# # Build the SoQL filter execution string
# # Filters for Virginia (VA), your specific county FIPS codes, and the metrics array
# where_clause = (
#     f"stateabbr = 'VA' AND "
#     f"locationid IN ({fips_in}) AND "
#     f"measureid IN ({measures_in})"
# )

# print(f"Executing batch pull from CDC PLACES API...")
# try:
#     # Query up to 2000 matching rows to handle multiple variables across counties
#     results = client.get(DATASET_ID, where=where_clause, limit=2000)
    
#     # Pack into a standard data analysis dataframe
#     df = pd.DataFrame.from_records(results)
    
#     if not df.empty:
#         # Isolate key metrics columns
#         # Note: Depending on the API release metadata, years are mapped to 'year' or 'datareleaseyear'
#         output_cols = ['locationname', 'countyfips', 'measureid', 'data_value', 'year']
#         available_cols = [c for c in output_cols if c in df.columns]
        
#         # Sort values cleanly by County Name and Metric Type
#         df_clean = df[available_cols].sort_values(by=['locationname', 'measureid'])
        
#         print(f"\nSuccessfully pulled {len(df_clean)} rows.")
#         print(df_clean.head(15).to_string(index=False))
        
#         # Save straight to your local multivariate directory workspace
#         df_clean.to_csv("central_va_cdc_places_metrics.csv", index=False)
#         print("\nSaved raw data matrix to 'central_va_cdc_places_metrics.csv'")
#     else:
#         print("API executed successfully but returned zero rows matching criteria.")
        
# except Exception as e:
#     print(f"API Endpoint Execution Failed: {e}")
# finally:
#     client.close()

# # Keep age-adjusted rows only (avoids duplicate jurisdiction × measureid rows)
# df_clean = df_clean[df_clean["data_value_type"] == "Age-adjusted prevalence"]

# # Pivot to one row per jurisdiction with one column per measure
# cdc = df_clean.pivot_table(
#     index="locationname",
#     columns="measureid",
#     values="data_value",
#     aggfunc="first"
# ).reset_index()

# # Rename columns to match pipeline convention
# cdc.columns.name = None
# cdc = cdc.rename(columns={
#     "locationname": "jurisdiction",
#     "BPHIGH":       "health_bphigh_pct",
#     "CASTHMA":      "health_casthma_pct",
#     "CSMOKING":     "health_csmoking_pct",
#     "DEPRESSION":   "health_depression_pct",
#     "DIABETES":     "health_diabetes_pct",
#     "MHLTH":        "health_mhlth_pct",
#     "OBESITY":      "health_obesity_pct",
#     "PHLTH":        "health_phlth_pct",
# })
# cdc["places_data_year"] = 2023



In [36]:
# HEALTH_MEASURES = [
#     "OBESITY", "DIABETES", "MHLTH", "PHLTH",
#     "CSMOKING", "BPHIGH", "CASTHMA", "DEPRESSION",
# ]

# fips_list = list(FIPS_CODES.values())
# fips_filter = " OR ".join([f"countyfips='{f}'" for f in fips_list])

# # Try 2024 release first, then fall back to alternate dataset ID
# cdc_urls = [
#     f"https://data.cdc.gov/resource/swc5-untb.json?$where={fips_filter}&$limit=500",
#     f"https://data.cdc.gov/resource/fu4u-a9bh.json?$where={fips_filter}&$limit=500",
#     f"https://data.cdc.gov/resource/d3i6-k6z5.json?$where={fips_filter}&$limit=500",
# ]

# cdc_raw = []
# for url in cdc_urls:
#     dest = OUTPUT_DIR / "cdc_places_virginia.json"
#     ok = download(url, dest, "CDC-PLACES")
#     if ok:
#         try:
#             data = json.loads(dest.read_text())
#             if data:
#                 cdc_raw = data
#                 log.info(f"[CDC] Retrieved {len(data)} records from {url}")
#                 break
#         except Exception as exc:
#             log.warning(f"[CDC] JSON parse failed: {exc}")

# if cdc_raw:
#     df_cdc = pd.DataFrame(cdc_raw)
#     log.info(f"[CDC] Columns: {list(df_cdc.columns)[:12]}")

#     fips_col  = next((c for c in df_cdc.columns if "fips" in c.lower()), None)
#     meas_col  = next((c for c in df_cdc.columns if "measure" in c.lower()), None)
#     val_col   = next((c for c in df_cdc.columns
#                       if "crude" in c.lower() and "prev" in c.lower()), None)
#     if val_col is None:
#         val_col = next((c for c in df_cdc.columns if "data_value" in c.lower()), None)

#     if fips_col and meas_col and val_col:
#         df_cdc[val_col] = pd.to_numeric(df_cdc[val_col], errors="coerce")
#         df_cdc["measure_short"] = df_cdc[meas_col].str.extract(
#             r"(" + "|".join(HEALTH_MEASURES) + r")", expand=False
#         )
#         df_filtered = df_cdc[df_cdc["measure_short"].notna()].copy()

#         cdc_pivot = df_filtered.pivot_table(
#             index=fips_col, columns="measure_short",
#             values=val_col, aggfunc="mean"
#         ).reset_index()
#         cdc_pivot.columns = [fips_col] + [
#             f"health_{c.lower()}_pct" for c in cdc_pivot.columns[1:]
#         ]

#         fips_to_juris = {v: k for k, v in FIPS_CODES.items()}
#         cdc_pivot["jurisdiction"] = cdc_pivot[fips_col].map(fips_to_juris)
#         cdc = cdc_pivot.dropna(subset=["jurisdiction"]).drop(columns=[fips_col])
#         cdc["places_data_year"] = 2024

#         print(f"\nCDC PLACES dataset: {len(cdc)} jurisdictions")
#         health_cols = [c for c in cdc.columns if "health_" in c]
#         print(f"Health measures: {health_cols}")
#         print(cdc[["jurisdiction"] + health_cols].to_string(index=False))
#     else:
#         log.warning(f"[CDC] Unexpected column structure. Saving raw for inspection.")
#         df_cdc.to_csv(OUTPUT_DIR / "cdc_places_raw.csv", index=False)
#         cdc = pd.DataFrame()
#         print("WARNING: CDC PLACES column structure unexpected. Check cdc_places_raw.csv")
# else:
#     cdc = pd.DataFrame()
#     print("WARNING: CDC PLACES data could not be retrieved.")

## Cell 8 — Download Census ACS Socioeconomic Controls

Fetches ACS 5-Year estimates for each study jurisdiction and year (2019–2024):
- Median household income (`B19013_001E`)
- Educational attainment — % bachelor's degree or higher
- Total population (for density calculation)

**Optional:** Set your Census API key to avoid rate limiting:
```python
import os; os.environ['CENSUS_API_KEY'] = 'your_key_here'
```
Free key at: https://api.census.gov/data/key_signup.html

In [37]:
ACS_VARIABLES = [
    "B19013_001E",   # median household income
    "B15003_001E",   # population 25+ (education denominator)
    "B15003_022E",   # bachelor's degree
    "B15003_023E",   # master's degree
    "B15003_024E",   # professional degree
    "B15003_025E",   # doctorate degree
    "B01003_001E",   # total population
]

acs_frames = []
acs_years = [y for y in STUDY_YEARS if y <= 2024]

for year in acs_years:
    var_string = ",".join(["NAME"] + ACS_VARIABLES)
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = f"get={var_string}&for=county:*&in=state:51"
    if CENSUS_API_KEY:
        params += f"&key={CENSUS_API_KEY}"
    url = f"{base_url}?{params}"

    dest = OUTPUT_DIR / f"acs5_{year}_virginia.json"
    ok = download(url, dest, f"ACS/{year}")
    if not ok:
        continue

    try:
        raw = json.loads(dest.read_text())
    except Exception as exc:
        log.warning(f"[ACS] {year}: JSON parse failed: {exc}")
        continue

    if len(raw) < 2:
        log.warning(f"[ACS] {year}: Empty response")
        continue

    df = pd.DataFrame(raw[1:], columns=raw[0])
    df = df[df["county"].isin(COUNTY_FIPS.values())].copy()
    if df.empty:
        log.warning(f"[ACS] {year}: No matching counties found")
        continue

    county_to_juris = {v: k for k, v in COUNTY_FIPS.items()}
    df["jurisdiction"] = df["county"].map(county_to_juris)
    df["year"] = year

    for col in ACS_VARIABLES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["median_household_income"] = df["B19013_001E"]
    df["total_population"]        = df["B01003_001E"]
    df["pct_bachelors_plus"] = (
        (df["B15003_022E"] + df["B15003_023E"] +
         df["B15003_024E"] + df["B15003_025E"])
        / df["B15003_001E"] * 100
    ).round(2)

    acs_frames.append(df[["jurisdiction", "year", "median_household_income",
                           "total_population", "pct_bachelors_plus"]])
    log.info(f"[ACS] {year}: {len(df)} jurisdictions loaded")

if acs_frames:
    acs = pd.concat(acs_frames, ignore_index=True)

    # Add population density using TIGER land area
    #tiger_url = "https://api.census.gov/data/2020/dec/pl?get=NAME,AREALAND&for=county:*&in=state:51"
    tiger_url = "https://api.census.gov/data/2023/geoinfo?get=NAME,AREALAND_SQMI&for=county:*&in=state:51"
    if CENSUS_API_KEY:
        tiger_url += f"&key={CENSUS_API_KEY}"
    dest_tiger = OUTPUT_DIR / "tiger_land_area_virginia.json"
    if download(tiger_url, dest_tiger, "TIGER/LandArea"):
        try:
            raw_tiger = json.loads(dest_tiger.read_text())
            land = pd.DataFrame(raw_tiger[1:], columns=raw_tiger[0])
            land = land[land["county"].isin(COUNTY_FIPS.values())].copy()
            county_to_juris = {v: k for k, v in COUNTY_FIPS.items()}
            land["jurisdiction"] = land["county"].map(county_to_juris)
            #land["land_area_sqmi"] = pd.to_numeric(land["AREALAND"], errors="coerce") / 2_589_988.11
            land["land_area_sqmi"] = pd.to_numeric(land["AREALAND_SQMI"], errors="coerce") / 2_589_988.11
            acs = acs.merge(land[["jurisdiction", "land_area_sqmi"]], on="jurisdiction", how="left")
            acs["pop_density_per_sqmi"] = (acs["total_population"] / acs["land_area_sqmi"]).round(2)
            acs = acs.drop(columns=["land_area_sqmi"])
            log.info("[ACS] Population density calculated.")
        except Exception as exc:
            log.warning(f"[ACS] Density calculation failed: {exc}")
            acs["pop_density_per_sqmi"] = float("nan")
    else:
        acs["pop_density_per_sqmi"] = float("nan")

    print(f"\nACS dataset: {len(acs)} rows")
    print(acs.pivot_table(index="jurisdiction", columns="year", values="median_household_income"))
else:
    acs = pd.DataFrame(columns=["jurisdiction", "year", "median_household_income",
                                 "total_population", "pct_bachelors_plus", "pop_density_per_sqmi"])
    print("WARNING: No ACS data retrieved. Set CENSUS_API_KEY and re-run.")

00:21:20  INFO      [ACS/2019] Downloading (attempt 1): https://api.census.gov/data/2019/acs/acs5?get=NAME,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B01003_001E&for=county:*&in=state:51&key=1d33172c35f206132f94362ee4786af3e6a82a62
00:21:20  INFO      [ACS/2019] Saved -> /home/ec2-user/SageMaker/policy_pipeline_output/acs5_2019_virginia.json  (12,274 bytes)
00:21:20  INFO      [ACS] 2019: 9 jurisdictions loaded
00:21:20  INFO      [ACS/2020] Downloading (attempt 1): https://api.census.gov/data/2020/acs/acs5?get=NAME,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B01003_001E&for=county:*&in=state:51&key=1d33172c35f206132f94362ee4786af3e6a82a62
00:21:21  INFO      [ACS/2020] Saved -> /home/ec2-user/SageMaker/policy_pipeline_output/acs5_2020_virginia.json  (12,277 bytes)
00:21:21  INFO      [ACS] 2020: 9 jurisdictions loaded
00:21:21  INFO      [ACS/2021] Downloading (attempt 1): https://api.census.gov/data/2021/acs/acs5?get=NAME,B1901


ACS dataset: 54 rows
year                2019     2020     2021     2022      2023      2024
jurisdiction                                                           
Albemarle        79880.0  84643.0  90568.0  97708.0  102617.0  104392.0
Augusta          62711.0  65076.0  69082.0  76124.0   79972.0   82049.0
Charlottesville  59471.0  59598.0  63470.0  67177.0   69829.0   74824.0
Fluvanna         76873.0  78885.0  82983.0  90766.0   91959.0   96768.0
Greene           67398.0  67266.0  73844.0  81338.0   85268.0   89808.0
Louisa           60975.0  67027.0  70974.0  76594.0   80343.0   86689.0
Nelson           64313.0  62203.0  67707.0  64028.0   68525.0   72589.0
Orange           71548.0  74446.0  79211.0  87309.0   94175.0   94008.0
Rockingham       61864.0  64496.0  67484.0  73232.0   78468.0   80693.0


## Cell 9 — Merge All Datasets into the Panel

Merges GEDI (spine) with SOL, crime, CDC PLACES, and ACS on `(jurisdiction, year)`.
CDC PLACES is cross-sectional and is broadcast across all years.

In [38]:
panel = gedi.copy()

# SOL pass rates
if not sol.empty:
    panel = panel.merge(sol, on=["jurisdiction", "year"], how="left")
    log.info(f"After SOL join: {len(panel)} rows, "
             f"{panel['sol_pass_rate_all'].notna().sum()} SOL values")

# Crime data
if not crime.empty:
    panel = panel.merge(crime, on=["jurisdiction", "year"], how="left")
    log.info(f"After crime join: {len(panel)} rows")

# ACS controls
if not acs.empty:
    panel = panel.merge(acs, on=["jurisdiction", "year"], how="left")
    log.info(f"After ACS join: {len(panel)} rows")

# CDC PLACES (cross-sectional — broadcast across all years)
if not cdc.empty:
    panel = panel.merge(cdc, on="jurisdiction", how="left")
    log.info(f"After CDC PLACES join: {len(panel)} rows")

# Add FIPS codes
panel["fips"] = panel["jurisdiction"].map(FIPS_CODES)

# Data quality flags
panel["flag_low_gedi_shots"] = (
    panel["total_valid_shots"].notna() & (panel["total_valid_shots"] < 1000)
)
if "sol_pass_rate_all" in panel.columns:
    panel["flag_sol_missing"] = panel["sol_pass_rate_all"].isna()
if "violent_crime_total" in panel.columns:
    panel["flag_crime_missing"] = panel["violent_crime_total"].isna()

panel = panel.sort_values(["jurisdiction", "year"]).reset_index(drop=True)

print(f"\nFinal panel dataset: {len(panel)} rows x {len(panel.columns)} columns")
print(f"Jurisdictions: {sorted(panel['jurisdiction'].unique())}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")
print(f"Columns: {list(panel.columns)}")
panel.head(16)

00:21:40  INFO      After SOL join: 68 rows, 47 SOL values
00:21:40  INFO      After crime join: 68 rows
00:21:40  INFO      After ACS join: 68 rows
00:21:40  INFO      After CDC PLACES join: 68 rows



Final panel dataset: 68 rows x 50 columns
Jurisdictions: ['Albemarle', 'Augusta', 'Buckingham', 'Charlottesville', 'Fluvanna', 'Greene', 'Louisa', 'Nelson', 'Orange', 'Rockingham']
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Columns: ['jurisdiction', 'year', 'canopy_height_mean_m', 'mean_canopy_cover', 'total_valid_shots', 'sol_pass_rate_all', 'agency_name', 'agency_code', 'population_estimate', 'incident_total', 'offense_total', 'group_a_crimes_per_100k', 'total_arrests', 'adult_arrests', 'juvenile_arrests', 'arrests_per_100k', 'total_group_a_offenses', 'total_group_b_adult_arrests', 'total_group_b_juvenile_arrests', 'aggravated_assault_reported', 'simple_assault_reported', 'burglary_breaking_entering_reported', 'destruction_damage_vandalism_reported', 'motor_vehicle_theft_reported', 'shoplifting_reported', 'theft_from_motor_vehicle_reported', 'all_other_larceny_reported', 'drug_narcotic_violations_reported',

,jurisdiction,year,canopy_height_mean_m,mean_canopy_cover,total_valid_shots,sol_pass_rate_all,agency_name,agency_code,population_estimate,incident_total,...,health_csmoking_pct,health_depression_pct,health_diabetes_pct,health_mhlth_pct,health_obesity_pct,health_phlth_pct,places_data_year,fips,flag_low_gedi_shots,flag_sol_missing
0,Albemarle,2019,19.675812,0.529159,53356,79.472,Albemarle County Police Department,VA0020300,109722.0,3324.0,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
1,Albemarle,2020,18.465603,0.540116,14835,NaN,Albemarle County Police Department,VA0020300,110545.0,3191.0,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,True
2,Albemarle,2021,18.793875,0.424221,54570,70.910,Albemarle County Police Department,VA0020300,114424.0,3454.0,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
3,Albemarle,2022,18.639362,0.459664,47618,67.400,NaN,NaN,NaN,NaN,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
4,Albemarle,2023,13.609988,0.207803,11110,71.156,NaN,NaN,NaN,NaN,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
5,Albemarle,2024,20.382057,0.512670,31573,73.058,NaN,NaN,NaN,NaN,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
6,Albemarle,2025,17.832790,0.404998,27296,73.338,NaN,NaN,NaN,NaN,...,9.6,21.8,8.7,15.2,29.0,10.5,2023.0,51003,False,False
7,Augusta,2019,14.742975,0.412462,23961,81.476,Augusta County Sheriff's Office,VA0080000,75831.0,2285.0,...,15.0,26.2,10.3,18.7,40.8,13.3,2023.0,51015,False,False
8,Augusta,2020,10.717906,0.233118,28070,NaN,Augusta County Sheriff's Office,VA0080000,76395.0,2142.0,...,15.0,26.2,10.3,18.7,40.8,13.3,2023.0,51015,False,True
9,Augusta,2021,14.507577,0.374634,33316,61.932,Augusta County Sheriff's Office,VA0080000,77598.0,1910.0,...,15.0,26.2,10.3,18.7,40.8,13.3,2023.0,51015,False,False


In [40]:
# Crime data
if not crime.empty:
    panel = panel.merge(crime, on=["jurisdiction", "year"], how="left")
    log.info(f"After crime join: {len(panel)} rows")

    # Build violent_crime_total from standard FBI UCR/NIBRS violent-offense
    # categories. Not assumed to already exist -- summed from whichever of
    # these component columns are actually present, since the crime CSV's
    # actual column names (per Cell 6) don't include a pre-built total.
    VIOLENT_CRIME_COMPONENTS = [
        "murder_reported", "homicide_reported", "rape_reported",
        "robbery_reported", "aggravated_assault_reported",
    ]
    present = [c for c in VIOLENT_CRIME_COMPONENTS if c in panel.columns]
    missing = [c for c in VIOLENT_CRIME_COMPONENTS if c not in panel.columns]

    if present:
        panel["violent_crime_total"] = panel[present].sum(axis=1, skipna=True)
        if missing:
            log.warning(f"[Crime] violent_crime_total built from {present} only -- "
                        f"missing standard components: {missing}. Treat as a "
                        f"partial proxy, not a complete UCR violent crime count.")
        else:
            log.info(f"[Crime] violent_crime_total built from all standard "
                      f"components: {present}")
    else:
        log.warning("[Crime] No recognized violent-crime component columns found "
                    "in the crime CSV; violent_crime_total not created. Print "
                    "list(crime.columns) to check actual column names.")

00:23:14  INFO      After crime join: 68 rows
00:23:14  WARNING   [Crime] violent_crime_total built from ['aggravated_assault_reported'] only -- missing standard components: ['murder_reported', 'homicide_reported', 'rape_reported', 'robbery_reported']. Treat as a partial proxy, not a complete UCR violent crime count.


## Cell 10 — Data Coverage Summary

In [41]:
import pandas as pd

outcome_cols = [
    "mean_canopy_cover",
    "canopy_height_mean_m",
    "sol_pass_rate_all",
    "violent_crime_total",
    "median_household_income",
    "pct_bachelors_plus",
    "pop_density_per_sqmi",
]
outcome_cols = [c for c in outcome_cols if c in panel.columns]

coverage = pd.DataFrame({
    "column": outcome_cols,
    "non_null": [panel[c].notna().sum() for c in outcome_cols],
    "total": len(panel),
})
coverage["coverage_pct"] = (coverage["non_null"] / coverage["total"] * 100).round(1)

print("Data Coverage Summary")
print("=" * 50)
print(coverage.to_string(index=False))

# Also show low-GEDI-shot flags
if "flag_low_gedi_shots" in panel.columns:
    low_shot_rows = panel[panel["flag_low_gedi_shots"] == True][["jurisdiction", "year", "total_valid_shots"]]
    if not low_shot_rows.empty:
        print(f"\nLow GEDI shot count rows (< 1000 shots):")
        print(low_shot_rows.to_string(index=False))

Data Coverage Summary
                 column  non_null  total  coverage_pct
      mean_canopy_cover        68     68         100.0
   canopy_height_mean_m        68     68         100.0
      sol_pass_rate_all        47     68          69.1
    violent_crime_total        68     68         100.0
median_household_income        52     68          76.5
     pct_bachelors_plus        52     68          76.5
   pop_density_per_sqmi        52     68          76.5

Low GEDI shot count rows (< 1000 shots):
   jurisdiction  year  total_valid_shots
Charlottesville  2019                407
Charlottesville  2021                472
Charlottesville  2022                528
Charlottesville  2024                160
Charlottesville  2025                 21
         Orange  2023                949


## Cell 11 — Save Outputs to Local and S3

Saves the panel dataset to `/tmp/policy_pipeline_output/` and uploads
to `s3://central-virginia-tree-canopy-project/dashboard-data/`.

In [42]:
S3_OUTPUT_DESTINATIONS = [
    ("central-virginia-tree-canopy-project", "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "data/"),
]

def upload_to_s3_all(local_path: Path, filename: str, label: str) -> None:
    """Upload a file to all three S3 destinations."""
    for bucket, prefix in S3_OUTPUT_DESTINATIONS:
        key = prefix + filename
        try:
            s3.upload_file(str(local_path), bucket, key)
            print(f"  -> s3://{bucket}/{key}")
        except Exception as exc:
            log.warning(f"  [WARN] Failed to upload to s3://{bucket}/{key}: {exc}")

# Save long-format panel CSV
long_path = OUTPUT_DIR / "policy_panel_dataset.csv"
panel.to_csv(long_path, index=False)
print(f"Saved locally: {long_path}")
print("Uploading policy_panel_dataset.csv to S3...")
upload_to_s3_all(long_path, "policy_panel_dataset.csv", "Panel CSV")

# Save wide-format panel CSV
wide_cols = [c for c in ["mean_canopy_cover", "canopy_height_mean_m",
                          "sol_pass_rate_all", "median_household_income"]
             if c in panel.columns]
if wide_cols:
    wide = panel.pivot_table(
        index="jurisdiction", columns="year", values=wide_cols, aggfunc="mean"
    )
    wide.columns = [f"{v}_{y}" for v, y in wide.columns]
    wide_path = OUTPUT_DIR / "policy_panel_dataset_wide.csv"
    wide.reset_index().to_csv(wide_path, index=False)
    print(f"\nSaved wide-format locally: {wide_path}")
    print("Uploading policy_panel_dataset_wide.csv to S3...")
    upload_to_s3_all(wide_path, "policy_panel_dataset_wide.csv", "Panel Wide CSV")

# Save panel as JSON for dashboard use
json_path = OUTPUT_DIR / "policy_panel_dataset.json"
panel.astype(object).where(panel.notna(), other=None).to_json(
    json_path, orient="records", indent=2
)
print(f"\nSaved JSON locally: {json_path}")
print("Uploading policy_panel_dataset.json to S3...")
upload_to_s3_all(json_path, "policy_panel_dataset.json", "Panel JSON")

print("\nAll outputs saved.")

Saved locally: /home/ec2-user/SageMaker/policy_pipeline_output/policy_panel_dataset.csv
Uploading policy_panel_dataset.csv to S3...
  -> s3://central-virginia-tree-canopy-project/dashboard-data/policy_panel_dataset.csv
  -> s3://central-va-tree-canopy-dashboard/dashboard-data/policy_panel_dataset.csv
  -> s3://central-va-tree-canopy-dashboard/data/policy_panel_dataset.csv

Saved wide-format locally: /home/ec2-user/SageMaker/policy_pipeline_output/policy_panel_dataset_wide.csv
Uploading policy_panel_dataset_wide.csv to S3...
  -> s3://central-virginia-tree-canopy-project/dashboard-data/policy_panel_dataset_wide.csv
  -> s3://central-va-tree-canopy-dashboard/dashboard-data/policy_panel_dataset_wide.csv
  -> s3://central-va-tree-canopy-dashboard/data/policy_panel_dataset_wide.csv

Saved JSON locally: /home/ec2-user/SageMaker/policy_pipeline_output/policy_panel_dataset.json
Uploading policy_panel_dataset.json to S3...
  -> s3://central-virginia-tree-canopy-project/dashboard-data/policy_pan

## Cell 12 — Fixed Effects Panel Regression

Runs a two-way Fixed Effects regression for each combination of:
- **Canopy variable:** `mean_canopy_cover`, `canopy_height_mean_m`
- **Outcome variable:** SOL pass rate, violent crime, obesity, diabetes, poor mental health

**Model:**
$$Y_{it} = \beta_1 C_{it} + \beta_2 X_{it} + \alpha_i + \gamma_t + \varepsilon_{it}$$

Requires: `pip install linearmodels statsmodels` (installed in Cell 1)

In [43]:
try:
    from linearmodels.panel import PanelOLS
    import statsmodels.api as sm
    HAS_LINEARMODELS = True
except ImportError:
    HAS_LINEARMODELS = False
    print("linearmodels not available. Run Cell 1 and restart the kernel.")

# Outcomes that are constant within jurisdiction across all years (CDC PLACES is
# cross-sectional, broadcast across years in Cell 9) -- two-way fixed effects
# cannot identify anything for these: entity-demeaning zeroes out the outcome
# entirely for every observation, since there's no within-jurisdiction variation
# to explain. These get a plain cross-sectional model instead.
CROSS_SECTIONAL_OUTCOMES = {"health_obesity_pct", "health_diabetes_pct", "health_mhlth_pct"}


def run_panel_model(df_reg, canopy_var, outcome_var, control_vars, outcome_label):
    req = [outcome_var, canopy_var] + control_vars
    df_m = df_reg[req].dropna()
    if len(df_m) < 10:
        log.warning(f"Insufficient data: {outcome_var} ~ {canopy_var} (n={len(df_m)})")
        return None

    y = df_m[outcome_var]
    X = sm.add_constant(df_m[[canopy_var] + control_vars])
    try:
        model  = PanelOLS(y, X, entity_effects=True, time_effects=True)
        result = model.fit(cov_type="clustered", cluster_entity=True)
    except Exception as exc:
        log.warning(f"Panel model failed: {outcome_var} ~ {canopy_var}: {exc}")
        return None

    pval = result.pvalues.get(canopy_var, float("nan"))
    return {
        "outcome":     outcome_label,
        "canopy_var":  canopy_var,
        "model_type":  "panel_FE (entity+time)",
        "beta":        round(result.params.get(canopy_var, float("nan")), 4),
        "p_value":     round(pval, 4),
        "r_squared":   round(result.rsquared, 3),
        "n_obs":       len(df_m),
        "significant": pval < 0.05,
    }


def run_cross_sectional_model(df_reg, canopy_var, outcome_var, control_vars, outcome_label):
    """
    For outcomes with no within-jurisdiction time variation. Collapses to one
    row per jurisdiction (mean canopy/controls across the study years, since
    the outcome itself is already a single value per jurisdiction) and runs
    an OLS with heteroskedasticity-robust (HC1) standard errors -- clustering
    isn't meaningful here with only ~9 jurisdictions as both the sample and
    the cluster count.
    """
    needed = [outcome_var, canopy_var] + control_vars
    cross = (
        df_reg.reset_index()
        .groupby("jurisdiction")[needed]
        .mean()
        .dropna()
    )
    if len(cross) < 5:
        log.warning(f"Insufficient cross-sectional data: {outcome_var} ~ {canopy_var} (n={len(cross)})")
        return None

    y = cross[outcome_var]
    X = sm.add_constant(cross[[canopy_var] + control_vars])
    try:
        result = sm.OLS(y, X).fit(cov_type="HC1")
    except Exception as exc:
        log.warning(f"Cross-sectional model failed: {outcome_var} ~ {canopy_var}: {exc}")
        return None

    pval = result.pvalues.get(canopy_var, float("nan"))
    return {
        "outcome":     outcome_label,
        "canopy_var":  canopy_var,
        "model_type":  "cross_sectional_OLS",
        "beta":        round(result.params.get(canopy_var, float("nan")), 4),
        "p_value":     round(pval, 4),
        "r_squared":   round(result.rsquared, 3),
        "n_obs":       int(result.nobs),
        "significant": pval < 0.05,
    }


if HAS_LINEARMODELS:
    df_reg = panel.copy()
    df_reg = df_reg.set_index(["jurisdiction", "year"])

    canopy_vars  = [c for c in ["mean_canopy_cover", "canopy_height_mean_m"] if c in df_reg.columns]
    control_vars = [c for c in ["median_household_income", "pop_density_per_sqmi",
                                 "pct_bachelors_plus"] if c in df_reg.columns]
    outcome_vars = {
        "sol_pass_rate_all":    "K-12 SOL Pass Rate (%)",
        "violent_crime_total":  "Violent Crime Total",
        "health_obesity_pct":   "Obesity Prevalence (%)",
        "health_diabetes_pct":  "Diabetes Prevalence (%)",
        "health_mhlth_pct":     "Poor Mental Health (%)",
    }
    outcome_vars = {k: v for k, v in outcome_vars.items() if k in df_reg.columns}

    results_rows = []
    for canopy_var in canopy_vars:
        for outcome_var, outcome_label in outcome_vars.items():
            if outcome_var in CROSS_SECTIONAL_OUTCOMES:
                row = run_cross_sectional_model(df_reg, canopy_var, outcome_var, control_vars, outcome_label)
            else:
                row = run_panel_model(df_reg, canopy_var, outcome_var, control_vars, outcome_label)
            if row:
                results_rows.append(row)

    if results_rows:
        results_df = pd.DataFrame(results_rows)

        reg_path = OUTPUT_DIR / "regression_results.csv"
        results_df.to_csv(reg_path, index=False)
        upload_to_s3_all(reg_path, "regression_results.csv", "Regression Results")

        print("\n" + "=" * 80)
        print("REGRESSION RESULTS (panel FE for time-varying outcomes, cross-sectional OLS for CDC PLACES)")
        print("=" * 80)
        print(results_df.to_string(index=False))
        print("=" * 80)

        sig = results_df[results_df["significant"] == True]
        if not sig.empty:
            print(f"\nStatistically significant findings (p < 0.05):")
            print(sig[["outcome", "canopy_var", "model_type", "beta", "p_value", "n_obs"]].to_string(index=False))
        else:
            print("\nNo statistically significant findings at p < 0.05.")
            print("Note: With N=8-9 jurisdictions and T<=7 years, statistical power is limited.")
            print("Consider this a pilot study requiring additional data collection.")
    else:
        print("No regression models could be estimated. Check data coverage in Cell 10.")

  -> s3://central-virginia-tree-canopy-project/dashboard-data/regression_results.csv
  -> s3://central-va-tree-canopy-dashboard/dashboard-data/regression_results.csv
  -> s3://central-va-tree-canopy-dashboard/data/regression_results.csv

REGRESSION RESULTS (panel FE for time-varying outcomes, cross-sectional OLS for CDC PLACES)
                outcome           canopy_var             model_type      beta  p_value  r_squared  n_obs  significant
 K-12 SOL Pass Rate (%)    mean_canopy_cover panel_FE (entity+time)   -8.5608   0.3351      0.251     39        False
    Violent Crime Total    mean_canopy_cover panel_FE (entity+time) -139.9651   0.0271      0.459     52         True
 Obesity Prevalence (%)    mean_canopy_cover    cross_sectional_OLS   11.0881   0.5655      0.721      9        False
Diabetes Prevalence (%)    mean_canopy_cover    cross_sectional_OLS    4.0585   0.0000      0.967      9         True
 Poor Mental Health (%)    mean_canopy_cover    cross_sectional_OLS    0.8643   

The Diabetes result (p = 0.0000, R² = 0.967, n = 9) — very likely overfitting, not a discovery
This is the one that should worry you most, precisely because it looks the most impressive. With n = 9 observations and 4 predictors (canopy + 3 controls) plus an intercept = 5 parameters, you have only 4 residual degrees of freedom. That's a well-known danger zone in regression: as the number of predictors approaches the number of observations, R² mechanically climbs toward 1 almost regardless of whether there's a real relationship — the model has enough free parameters to trace through the data points almost exactly. A rule of thumb econometricians use is roughly 10-20 observations per predictor for OLS to be trustworthy; here you have about 2 observations per predictor.
The p = 0.0000 compounds the concern rather than resolving it — with only 4 residual df, that p-value is riding on a razor-thin amount of information, and is highly sensitive to any single jurisdiction being an outlier or high-leverage point. A few concrete things worth checking before trusting this at all.....

If VIF values are large (>10, easily possible with 4 correlated-ish socioeconomic predictors and only 9 points) or one jurisdiction has a Cook's distance well above the others, that R²=0.967 is explained by overfitting/leverage, not a real diabetes-canopy relationship.

In [44]:
#These results need to be checked

# 1. Check for multicollinearity among your 3 controls + canopy var (n=9 makes this likely severe)
from statsmodels.stats.outliers_influence import variance_inflation_factor
cross = df_reg.reset_index().groupby("jurisdiction")[["health_diabetes_pct", "mean_canopy_cover",
    "median_household_income", "pop_density_per_sqmi", "pct_bachelors_plus"]].mean().dropna()
X_vif = sm.add_constant(cross[["mean_canopy_cover", "median_household_income",
                                 "pop_density_per_sqmi", "pct_bachelors_plus"]])
for i, col in enumerate(X_vif.columns):
    print(col, variance_inflation_factor(X_vif.values, i))

# 2. Check influence/leverage -- is one jurisdiction (e.g. Charlottesville, the
#    outlier-prone one per your own flag_low_gedi_shots note) driving everything?
result = sm.OLS(cross["health_diabetes_pct"], X_vif).fit()
print(result.get_influence().summary_frame()[["cooks_d", "hat_diag"]])

const 233.44958061516482
mean_canopy_cover 1.5137303213652282
median_household_income 2.9438301568696295
pop_density_per_sqmi 5.365629542360654
pct_bachelors_plus 5.353635075744768
                    cooks_d  hat_diag
jurisdiction                         
Albemarle          3.292645  0.948604
Augusta            0.027771  0.412312
Charlottesville  367.708568  0.999892
Fluvanna           0.051610  0.398759
Greene             0.001381  0.157762
Louisa             0.163626  0.182556
Nelson             4.650289  0.893624
Orange             0.053651  0.457201
Rockingham         0.059986  0.549290


## Findings 
Multicollinearity: not the main issue
VIFs for the actual predictors are all under 10 (mean_canopy_cover: 1.51, median_household_income: 2.94, pop_density_per_sqmi: 5.37, pct_bachelors_plus: 5.35). Density and education are moderately correlated with each other (unsurprising — denser places tend to be more educated), but nothing here rises to a "this alone breaks the model" level. The const VIF of 233 is not meaningful — that's just an artifact of the intercept's VIF when predictors aren't centered, not a real collinearity signal.
The real culprit: Charlottesville has a hat value of 0.9999 and Cook's D of 367.7
This is the finding. For context: with n=9 and 5 parameters (intercept + 4 predictors), average leverage is 5/9 ≈ 0.56. Charlottesville's leverage is 0.9999 — essentially 1, the mathematical maximum. A hat value that close to 1 means the model can fit that single point almost exactly, contributing nearly zero to the residual sum of squares. And the standard rule-of-thumb Cook's distance concern threshold is around 4/n ≈ 0.44 — Charlottesville's value of 367.7 isn't just above that threshold, it's roughly 800× larger.
Translation: with only 9 jurisdictions and this many predictors, the regression is essentially spending almost all of its 4 degrees of freedom perfectly threading itself through Charlottesville's single data point, leaving barely any real information left to test whether canopy cover and diabetes prevalence are actually related across the other 8. That p = 0.0000, R² = 0.967 result is very likely an artifact of one extreme, high-leverage observation — not a real population relationship. This lines up with something your own pipeline already flags: Charlottesville is structurally different from the other 8 (a small, dense independent city vs. rural/suburban counties) and already carries a flag_low_gedi_shots warning in several years for a different reason (small geographic footprint, limited GEDI orbital coverage) — it's a genuine outlier in this study design, not noise.
The definitive test — leave-one-out:


In [45]:
# Does the "significant" diabetes result survive without Charlottesville?
cross_loo = cross.drop(index="Charlottesville")
y_loo = cross_loo["health_diabetes_pct"]
X_loo = sm.add_constant(cross_loo[["mean_canopy_cover", "median_household_income",
                                     "pop_density_per_sqmi", "pct_bachelors_plus"]])
result_loo = sm.OLS(y_loo, X_loo).fit(cov_type="HC1")
print(f"n={int(result_loo.nobs)}, beta={result_loo.params['mean_canopy_cover']:.4f}, "
      f"p={result_loo.pvalues['mean_canopy_cover']:.4f}, R²={result_loo.rsquared:.3f}")

n=8, beta=4.5261, p=0.0000, R²=0.952


I am not completely sure but I would bet strongly that with Charlottesville removed (down to n=8, 3 residual df — already fragile territory, but informative as a check), either the p-value stops being anywhere near 0.05, or the coefficient's sign/magnitude changes substantially. Either outcome tells you the original "finding" was Charlottesville-driven rather than a real cross-jurisdiction pattern.
Also worth a quick look, since they're not nothing either: Albemarle (Cook's D = 3.29) and Nelson (4.65) are both well above the 0.44 threshold too, just not in Charlottesville's league. Worth running the same leave-one-out check for those if the Charlottesville-excluded model still looks interesting, rather than assuming the rest of the sample is clean.
Given all this — n=9, one point at essentially the mathematical maximum leverage, and a p-value that's likely non-robust to dropping that single observation — I'd recommend not reporting the Diabetes finding as a real result in anything downstream. It's a solid example of exactly why the pipeline's own "pilot study, hypothesis-generating only" framing exists.

## A quick test

In [26]:
# try:
#     from linearmodels.panel import PanelOLS
#     import statsmodels.api as sm
#     HAS_LINEARMODELS = True
# except ImportError:
#     HAS_LINEARMODELS = False
#     print("linearmodels not available. Run Cell 1 and restart the kernel.")

# if HAS_LINEARMODELS:
#     df_reg = panel.copy()
#     df_reg = df_reg.set_index(["jurisdiction", "year"])

#     canopy_vars  = [c for c in ["mean_canopy_cover", "canopy_height_mean_m"] if c in df_reg.columns]
#     control_vars = [c for c in ["median_household_income", "pop_density_per_sqmi",
#                                  "pct_bachelors_plus"] if c in df_reg.columns]
#     outcome_vars = {
#         "sol_pass_rate_all":    "K-12 SOL Pass Rate (%)",
#         "violent_crime_total":  "Violent Crime Total",
#         "health_obesity_pct":   "Obesity Prevalence (%)",
#         "health_diabetes_pct":  "Diabetes Prevalence (%)",
#         "health_mhlth_pct":     "Poor Mental Health (%)",
#     }
#     outcome_vars = {k: v for k, v in outcome_vars.items() if k in df_reg.columns}

#     results_rows = []

#     for canopy_var in canopy_vars:
#         for outcome_var, outcome_label in outcome_vars.items():
#             req = [outcome_var, canopy_var] + control_vars
#             df_m = df_reg[req].dropna()

#             if len(df_m) < 10:
#                 log.warning(f"Insufficient data: {outcome_var} ~ {canopy_var} (n={len(df_m)})")
#                 continue

#             y = df_m[outcome_var]
#             X = sm.add_constant(df_m[[canopy_var] + control_vars])

#             try:
#                 model  = PanelOLS(y, X, entity_effects=True, time_effects=True)
#                 result = model.fit(cov_type="clustered", cluster_entity=True)

#                 coef = result.params.get(canopy_var, float("nan"))
#                 pval = result.pvalues.get(canopy_var, float("nan"))
#                 r2   = result.rsquared

#                 results_rows.append({
#                     "outcome":     outcome_label,
#                     "canopy_var":  canopy_var,
#                     "beta":        round(coef, 4),
#                     "p_value":     round(pval, 4),
#                     "r_squared":   round(r2, 3),
#                     "n_obs":       len(df_m),
#                     "significant": pval < 0.05,
#                 })
#             except Exception as exc:
#                 log.warning(f"Model failed: {outcome_var} ~ {canopy_var}: {exc}")

#     if results_rows:
#         results_df = pd.DataFrame(results_rows)

#         # Save locally and upload to S3
#         reg_path = OUTPUT_DIR / "regression_results.csv"
#         results_df.to_csv(reg_path, index=False)
#         upload_to_s3_all(reg_path, "regression_results.csv", "Regression Results")

#         print("\n" + "=" * 70)
#         print("FIXED EFFECTS REGRESSION RESULTS")
#         print("Model: Y_it = beta1*C_it + beta2*X_it + alpha_i + gamma_t + eps_it")
#         print("=" * 70)
#         print(results_df.to_string(index=False))
#         print("=" * 70)

#         sig = results_df[results_df["significant"] == True]
#         if not sig.empty:
#             print(f"\nStatistically significant findings (p < 0.05):")
#             print(sig[["outcome", "canopy_var", "beta", "p_value", "n_obs"]].to_string(index=False))
#         else:
#             print("\nNo statistically significant findings at p < 0.05.")
#             print("Note: With N=8 jurisdictions and T<=7 years, statistical power is limited.")
#             print("Consider this a pilot study requiring additional data collection.")
#     else:
#         print("No regression models could be estimated. Check data coverage in Cell 10.")

  -> s3://central-virginia-tree-canopy-project/dashboard-data/regression_results.csv
  -> s3://central-va-tree-canopy-dashboard/dashboard-data/regression_results.csv
  -> s3://central-va-tree-canopy-dashboard/data/regression_results.csv

FIXED EFFECTS REGRESSION RESULTS
Model: Y_it = beta1*C_it + beta2*X_it + alpha_i + gamma_t + eps_it
                outcome           canopy_var    beta  p_value  r_squared  n_obs  significant
 K-12 SOL Pass Rate (%)    mean_canopy_cover -8.5608   0.3351      0.251     39        False
 Obesity Prevalence (%)    mean_canopy_cover -0.0000   0.4625      0.000     52        False
Diabetes Prevalence (%)    mean_canopy_cover -0.0000   0.2552     -6.500     52        False
 Poor Mental Health (%)    mean_canopy_cover -0.0000   0.5082     -9.500     52        False
 K-12 SOL Pass Rate (%) canopy_height_mean_m -0.0400   0.8860      0.232     39        False
 Obesity Prevalence (%) canopy_height_mean_m  0.0000   0.8841      0.000     52        False
Diabetes Pr

## Notes and Limitations

### Data Quality Flags
- **2019 SMAP anomaly:** The 2019 SMAP soil moisture value covers only 32 days (Jan–Feb). Treat with caution in any temporal analysis.
- **Charlottesville GEDI shots:** Charlottesville City has very low GEDI shot counts in some years (as few as 21 shots in 2025) due to its small geographic footprint. Rows flagged with `flag_low_gedi_shots = True` should be excluded from regression models or treated as outliers.
- **2023 canopy cover drop:** All jurisdictions show a sharp drop in `mean_canopy_cover` in 2023. This may reflect a satellite coverage gap, drought stress, or real deforestation. Investigate before including 2023 in regression models.

### Statistical Power
With N=8 jurisdictions and T=5–7 years, the panel has at most 56 observations. After removing missing values and applying fixed effects, the effective degrees of freedom are very limited. The regression results should be interpreted as **exploratory/hypothesis-generating**, not as definitive causal evidence.

### Crime Data
Add instructions here 

### CDC PLACES
CDC PLACES county data is cross-sectional (one estimate per county, not per year). It is broadcast across all study years in this pipeline. For longitudinal health analysis, consider the CDC PLACES trend data or BRFSS county-level estimates.